# Trigger-Driven Validation POC
trigger_*.ts + page_*.ts 코드를 읽어 CDMSAgent 액션을 자동 생성하고 실행합니다.

**파이프라인**
```
TS 파일 → Node.js 추출 → conditional 구조 파싱 → 테스트값 생성 → CDMSAgent 실행 → FAIL 수집
```

## 1. CRF 스펙 추출 (TS → JSON)

In [ ]:
import subprocess, json, re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from datetime import date, timedelta
import pandas as pd

MAVEN_CRF_ROOT = Path(r'C:\Users\SunbeomGwon\maven-crfs')
STUDY = '20260105_FAST_AF'

result = subprocess.run(
    ['npx', 'ts-node', '--project', 'tsconfig.json',
     'scripts/extract_crf_spec.ts', STUDY],
    cwd=MAVEN_CRF_ROOT,
    capture_output=True, text=True, timeout=90
)
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])
    raise RuntimeError('extractor 실패')

spec = json.loads(result.stdout)
print(f"Pages : {list(spec['pages'].keys())}")
print(f"Triggers: {len(spec['triggers'])}개 (QUERY only)")

## 2. 필드 정의 빌드 — itemId → (label, type, CDMSAgent action)

In [ ]:
@dataclass
class FieldDef:
    item_id: str
    label: str
    field_type: str        # DATE | TEXT | AUTO_TEXT | SINGLE_SELECT | CHECK
    layout: Optional[str]  # RADIO | CHECKBOX | None
    page_id: str
    section_id: str
    options: list = field(default_factory=list)

    @property
    def agent_action(self) -> str:
        if self.field_type == 'DATE':                              return 'set_date'
        if self.field_type == 'AUTO_TEXT':                         return 'SKIP'
        if self.field_type == 'SINGLE_SELECT' and self.layout == 'RADIO': return 'select_radio'
        if self.field_type in ('SINGLE_SELECT', 'CHECK'):          return 'select_option'
        return 'set_text'

# itemId → FieldDef  (페이지.아이템 복합키로 중복 방지)
FIELD_MAP: dict[str, FieldDef] = {}
for page_id, fields in spec['pages'].items():
    for f in fields:
        if not f.get('itemId'):
            continue
        fd = FieldDef(
            item_id=f['itemId'], label=f['label'], field_type=f['type'],
            layout=f.get('layout'), page_id=page_id, section_id=f['sectionId'],
            options=f.get('options', []),
        )
        FIELD_MAP[f"{page_id}.{f['itemId']}"] = fd
        FIELD_MAP.setdefault(f['itemId'], fd)  # 단순 itemId 조회 (첫 등장 우선)

total = len([k for k in FIELD_MAP if '.' in k])
print(f"FieldDef {total}개 로드")
rows = [{'key': k, 'label': v.label, 'type': v.field_type,
          'layout': v.layout, 'action': v.agent_action}
         for k, v in FIELD_MAP.items() if '.' in k]
pd.DataFrame(rows).head(25)

## 3. conditional 구조 파싱 → 테스트 입력값 생성

| 분류 | 예 | 처리 |
|------|----|-----------|
| numeric | AGE < 19 | 경계값 자동 생성 |
| cross_field_same_visit | DIABP >= SYSBP | 두 필드 같은 방문 입력 |
| cross_visit_date | V3.SVDTC ≤ V2.SVDTC | 방문 간 날짜 계산 |
| date_arithmetic | V3-V1 days < 16 | 일수 오프셋 계산 |
| dynamic_visit (reserved:CURRENT) | UV 방문 반복 | SKIP |
| external_reserved | IRB_DATE | SKIP |

In [ ]:
@dataclass
class TestInput:
    item_id: str
    value: str
    visit_id: str = ''
    page_id: str = ''
    is_precondition: bool = False

@dataclass
class TestCase:
    trigger_id: str
    note: str
    page_id: str
    issue_item_ids: list
    inputs: list
    expected: str = 'Query'
    skip_reason: str = ''

    @property
    def skipped(self): return bool(self.skip_reason)


# ── 헬퍼 ──────────────────────────────────────────────────────────────────────
def _boundary(op: str, threshold: float) -> str:
    """op + threshold → 조건을 위반(쿼리를 발생시키는) 값"""
    if op == '<':  return str(threshold - 1)
    if op == '>':  return str(threshold + 1)
    if op == '<=': return str(threshold - 1)  # 같거나 작음 → 더 작게
    if op == '>=': return str(threshold + 1)  # 같거나 크면 → 더 크게
    if op == '!=': return str(threshold)       # 달라야 함 → 같게
    if op == '=':  return str(threshold + 1)   # 같아야 함 → 다르게
    return str(threshold)

def _date(delta_days: int = 0) -> str:
    return (date.today() + timedelta(days=delta_days)).isoformat()

def _is_null_guard(expr: dict) -> bool:
    """e.g. {left:..., operator:'!=', right:null} → not-null 가드, 파싱 스킵"""
    return expr.get('right') is None and expr.get('operator') == '!='

def _get_visit(ref: dict) -> str:
    v = ref.get('visitId', '')
    if isinstance(v, dict):  return ''  # {reserved: 'EACH'} 등
    return v or ''


# ── 핵심 파서 ─────────────────────────────────────────────────────────────────
def parse_conditional(cond: dict, trigger_page: str) -> tuple[list[TestInput], str]:
    """
    conditional 트리 → (TestInput 목록, skip_reason)
    skip_reason 비어 있으면 자동 처리 가능.
    """
    if not cond:
        return [], 'conditional 없음'

    inputs: list[TestInput] = []
    skip_reason = ''

    def walk(node: dict):
        nonlocal skip_reason
        if 'expr' in node:
            for child in node['expr']:
                walk(child)
            return

        left = node.get('left', {})
        op   = node.get('operator', '')
        right = node.get('right')

        # not-null 가드 → 스킵
        if _is_null_guard(node):
            return

        # ── CASE 1: right 가 시스템 예약값 (IRB, CONTRACT 등) ──
        if isinstance(right, dict) and right.get('reserved') and not right.get('itemId'):
            skip_reason = f"외부 시스템값: {right.get('reserved')}"
            return

        # ── CASE 2: 동적 방문 (reserved:CURRENT) ──
        if isinstance(left.get('visitId'), dict) or isinstance(right, dict) and isinstance(right.get('visitId'), dict):
            skip_reason = '동적 방문 ID (UV 반복 등)'
            return

        # ── CASE 3: date arithmetic — left 가 {valAs:'DAYS', left:ref, op:'-', right:ref} ──
        if isinstance(left, dict) and left.get('valAs') == 'DAYS' and isinstance(right, (int, float)):
            # (A_date - B_date) days  <op>  threshold
            a_ref = left.get('left', {})
            b_ref = left.get('right', {})
            a_item = a_ref.get('itemId', '')
            b_item = b_ref.get('itemId', '')
            a_visit = _get_visit(a_ref)
            b_visit = _get_visit(b_ref)
            threshold = float(right)
            # (A - B) < threshold → 트리거: A = B + (threshold - 1)일  (경계 내부)
            # (A - B) > threshold → 트리거: A = B + (threshold + 1)일
            if op == '<':
                # A - B < threshold  →  A = B + threshold - 1 (조건 만족)
                offset = int(threshold) - 1
            elif op == '>':
                offset = int(threshold) + 1
            elif op == '<=':
                offset = int(threshold) - 1
            else:
                offset = int(threshold)
            b_date = _date(0)   # B를 오늘로
            a_date = _date(offset)
            inputs.append(TestInput(item_id=b_item, value=b_date,
                                     visit_id=b_visit, page_id=b_ref.get('crfPageId',''),
                                     is_precondition=True))
            inputs.append(TestInput(item_id=a_item, value=a_date,
                                     visit_id=a_visit, page_id=a_ref.get('crfPageId','')))
            return

        # ── CASE 4: right 가 숫자 ──
        if isinstance(right, (int, float)):
            item_id = left.get('itemId', '')
            visit_id = _get_visit(left)
            page_id  = left.get('crfPageId', trigger_page)
            val = _boundary(op, float(right))
            inputs.append(TestInput(item_id=item_id, value=val,
                                     visit_id=visit_id, page_id=page_id))
            return

        # ── CASE 5: right 가 다른 아이템 참조 ──
        if isinstance(right, dict) and right.get('itemId'):
            l_item   = left.get('itemId', '')
            r_item   = right.get('itemId', '')
            l_visit  = _get_visit(left)
            r_visit  = _get_visit(right)
            l_page   = left.get('crfPageId', trigger_page)
            r_page   = right.get('crfPageId', trigger_page)
            l_type   = FIELD_MAP.get(l_item) and FIELD_MAP[l_item].field_type

            if l_type == 'DATE':
                # 날짜 비교: right(선행조건) 오늘, left를 op 위반 날짜로
                ref_date = _date(0)
                if op in ('<', '<='):   test_date = _date(-10)
                elif op in ('>', '>='): test_date = _date(10)
                else:                   test_date = _date(5)
                inputs.append(TestInput(item_id=r_item, value=ref_date,
                                         visit_id=r_visit, page_id=r_page,
                                         is_precondition=True))
                inputs.append(TestInput(item_id=l_item, value=test_date,
                                         visit_id=l_visit, page_id=l_page))
            else:
                # 숫자형 크로스 필드: right를 기준값, left를 위반값으로
                base = 100
                if op in ('<', '<='):   l_val, r_val = base - 1, base
                elif op in ('>', '>='): l_val, r_val = base + 1, base
                else:                   l_val, r_val = base, base
                inputs.append(TestInput(item_id=r_item, value=str(r_val),
                                         visit_id=r_visit, page_id=r_page,
                                         is_precondition=True))
                inputs.append(TestInput(item_id=l_item, value=str(l_val),
                                         visit_id=l_visit, page_id=l_page))
            return

    walk(cond)

    # 중복 제거 (같은 item_id+visit_id)
    seen = set()
    deduped = []
    for inp in inputs:
        key = (inp.item_id, inp.visit_id)
        if key not in seen:
            seen.add(key)
            deduped.append(inp)

    return deduped, skip_reason


def build_test_cases(spec: dict) -> list[TestCase]:
    cases = []
    for t in spec['triggers']:
        inputs, skip = parse_conditional(t['conditional'], t['pageId'])
        if not inputs and not skip:
            skip = '입력값 없음 (not-null 가드만 존재)'
        cases.append(TestCase(
            trigger_id=t['id'], note=t['note'], page_id=t['pageId'],
            issue_item_ids=t['issueItemIds'],
            inputs=inputs, skip_reason=skip,
        ))
    return cases


test_cases = build_test_cases(spec)
auto    = [c for c in test_cases if not c.skipped]
skipped = [c for c in test_cases if c.skipped]
print(f"자동 처리 가능 : {len(auto)}개")
print(f"SKIP          : {len(skipped)}개")
print(f"전체          : {len(test_cases)}개")

## 4. Dry-run — 실제로 어떤 액션이 생성되는지 확인

In [ ]:
@dataclass
class PlannedAction:
    trigger_id: str
    note: str
    item_id: str
    label: str
    action: str
    value: str
    visit_id: str
    page_id: str
    is_precondition: bool


def plan_actions(test_cases: list) -> list[PlannedAction]:
    planned = []
    for tc in test_cases:
        if tc.skipped:
            continue
        for inp in tc.inputs:
            fd = FIELD_MAP.get(f"{inp.page_id}.{inp.item_id}") or FIELD_MAP.get(inp.item_id)
            if fd is None:
                action, label = 'UNKNOWN_FIELD', inp.item_id
            elif fd.agent_action == 'SKIP':
                action, label = 'SKIP(AUTO_CALC)', fd.label
            else:
                action, label = fd.agent_action, fd.label
            planned.append(PlannedAction(
                trigger_id=tc.trigger_id, note=tc.note,
                item_id=inp.item_id, label=label, action=action,
                value=inp.value, visit_id=inp.visit_id, page_id=inp.page_id,
                is_precondition=inp.is_precondition,
            ))
    return planned


planned = plan_actions(test_cases)

df = pd.DataFrame([vars(p) for p in planned])
print(f"총 액션 수: {len(df)}")
print("\n액션 유형 분포:")
print(df['action'].value_counts().to_string())
df[['trigger_id','note','item_id','label','action','value','visit_id','is_precondition']]

In [ ]:
# SKIP된 트리거 목록
skip_rows = [{'trigger_id': c.trigger_id, 'note': c.note, 'reason': c.skip_reason}
             for c in skipped]
print(f"SKIP 트리거 {len(skip_rows)}개")
pd.DataFrame(skip_rows)

## 5. CDMSAgent 연결 확인

In [ ]:
from cdm_agent_client import CDMSAgent
from cdm_agent_client.exceptions import StepFailedError

agent = CDMSAgent(study_id='FAST_AF')
ok = agent.ping()
print('Daemon 연결:', ok)
if ok:
    snap = agent.inspect()
    print('현재 페이지:', snap.page_label, '|', snap.url)

## 6. 단일 트리거 실행 테스트

In [ ]:
def _ui_val(fd: Optional[FieldDef], raw_val: str) -> str:
    """dbVal → uiVal 변환 (라디오/콤보박스용)"""
    if fd is None: return raw_val
    for opt in fd.options:
        if str(opt.get('dbVal', '')) == str(raw_val):
            return opt['uiVal']
    return raw_val


def run_trigger_test(tc: TestCase, planned: list[PlannedAction]) -> dict:
    actions = [p for p in planned if p.trigger_id == tc.trigger_id]
    if not actions:
        return {'trigger_id': tc.trigger_id, 'note': tc.note,
                'result': 'SKIP', 'errors': ['no actions']}

    errors = []

    # 선행조건 먼저 (다른 페이지/방문)
    preconditions = [a for a in actions if a.is_precondition]
    main_actions  = [a for a in actions if not a.is_precondition]

    for act in preconditions:
        nav_seg = f"{act.visit_id}/{act.page_id}" if act.visit_id else act.page_id
        agent.go_to_page(nav_seg)
        _execute_action(act, errors)
        agent.click_save()

    # 메인 페이지로 이동
    if main_actions:
        nav_seg = main_actions[0].page_id
        agent.go_to_page(nav_seg)

    for act in main_actions:
        _execute_action(act, errors)

    # Save
    agent.click_save()

    # 결과 확인
    result = agent.check_result('Query')
    return {'trigger_id': tc.trigger_id, 'note': tc.note,
            'result': result, 'errors': errors}


def _execute_action(act: PlannedAction, errors: list) -> None:
    fd = FIELD_MAP.get(f"{act.page_id}.{act.item_id}") or FIELD_MAP.get(act.item_id)
    try:
        if act.action == 'set_date':
            agent.set_date(act.label, act.value)
        elif act.action == 'set_text':
            agent.set_text(act.label, act.value)
        elif act.action == 'select_radio':
            agent.select_radio(act.label, _ui_val(fd, act.value))
        elif act.action == 'select_option':
            agent.select_option(act.label, _ui_val(fd, act.value))
        # SKIP(AUTO_CALC), UNKNOWN_FIELD → 무시
    except StepFailedError as e:
        # fallback: text 입력 시도
        try:
            agent.set_text(act.label, act.value)
        except Exception:
            errors.append(f"{act.item_id}: {e}")


# 단일 케이스 테스트 — DM 페이지 AGE<19
target = next((c for c in auto if c.trigger_id == 'D_DM_SQ_2'), auto[0])
print(f"테스트: {target.trigger_id} | {target.note}")
print(f"입력값: {[(i.item_id, i.value) for i in target.inputs]}")
out = run_trigger_test(target, planned)
print(out)

## 7. 전체 자동 실행 — FAIL 케이스만 수집

In [ ]:
results = []
for tc in auto:
    try:
        r = run_trigger_test(tc, planned)
        results.append(r)
        mark = '✓' if r['result'] == 'PASS' else '✗'
        print(f"[{mark}] {tc.trigger_id} | {tc.note[:45]}")
    except Exception as e:
        results.append({'trigger_id': tc.trigger_id, 'note': tc.note,
                        'result': 'ERROR', 'errors': [str(e)]})
        print(f"[!] {tc.trigger_id} — {e}")

print(f"\n{'='*50}")
print(f"PASS  : {sum(1 for r in results if r['result'] == 'PASS')}")
print(f"FAIL  : {sum(1 for r in results if r['result'] == 'FAIL')}")
print(f"ERROR : {sum(1 for r in results if r['result'] == 'ERROR')}")
print(f"SKIP  : {len(skipped)} (파싱 불가)")

fails = [r for r in results if r['result'] != 'PASS']
pd.DataFrame(fails)[['trigger_id','note','result','errors']]